In [24]:
import pandas as pd
import json
import os
import math
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import defaultdict


In [25]:
# Define file paths
PASSAGE_COLLECTION_PATH = "../data/nq/raw/corpus.jsonl"
QUERY_FILE_PATH = "../data/nq/raw/queries.jsonl"
QRELS_FILE_PATH = "../data/nq/raw/test.tsv"

passages_df = pd.read_json(PASSAGE_COLLECTION_PATH, lines=True)
queries_df = pd.read_json(QUERY_FILE_PATH, lines=True)
qrels_df = pd.read_csv(QRELS_FILE_PATH, sep="\t")

In [26]:
print(f"Total number of passages: {len(passages_df)}")
print(f"Total number of queries: {len(queries_df)}")
print(f"Total number of qrels: {len(qrels_df)}")

Total number of passages: 2681468
Total number of queries: 3452
Total number of qrels: 4201


In [27]:
print(f"Passages dataframe head:\n{passages_df.head()}")
print(f"Queries dataframe head:\n{queries_df.head()}")
print(f"Qrels dataframe head:\n{qrels_df.head()}")

Passages dataframe head:
    _id              title                                               text  \
0  doc0  Minority interest  In accounting, minority interest (or non-contr...   
1  doc1  Minority interest  It is, however, possible (such as through spec...   
2  doc2  Minority interest  The reporting of 'minority interest' is a cons...   
3  doc3  Minority interest  Some investors have expressed concern that the...   
4  doc4  Minority interest  Minority interest is an integral part of the e...   

  metadata  
0       {}  
1       {}  
2       {}  
3       {}  
4       {}  
Queries dataframe head:
     _id                                               text metadata
0  test0  what is non controlling interest on balance sheet       {}
1  test1     how many episodes are in chicago fire season 4       {}
2  test2    who sings love will keep us alive by the eagles       {}
3  test3          who is the leader of the ontario pc party       {}
4  test4    nitty gritty dirt band fishin

In [28]:
# Rename passage dataframe and keep only relevant columns
passages_df = passages_df.rename(columns={
    "_id": "pid",
    "text": "passage"
})[["pid", "passage"]]  # keep only relevant columns

passages_df.head()

,pid,passage
0,doc0,"In accounting, minority interest (or non-contr..."
1,doc1,"It is, however, possible (such as through spec..."
2,doc2,The reporting of 'minority interest' is a cons...
3,doc3,Some investors have expressed concern that the...
4,doc4,Minority interest is an integral part of the e...


In [29]:
# Rename queries df and keep only relevant columns
queries_df = queries_df.rename(columns={
    "_id": "qid",
    "text": "query"
})[["qid", "query"]]

queries_df.head()

,qid,query
0,test0,what is non controlling interest on balance sheet
1,test1,how many episodes are in chicago fire season 4
2,test2,who sings love will keep us alive by the eagles
3,test3,who is the leader of the ontario pc party
4,test4,nitty gritty dirt band fishin in the dark album


In [30]:
# Rename qrels df and keep only relevant columns
qrels_df = qrels_df.rename(columns={
    "query-id": "qid",
    "corpus-id": "pid"
})[["qid", "pid"]]

qrels_df.head()

,qid,pid
0,test0,doc0
1,test0,doc1
2,test1,doc6
3,test2,doc10
4,test3,doc17


In [31]:
# Check if the `qid` in the qrels df is unique (if the len of unique `qid` equals to the len of the df)
# If they are not unique, means that there are multiple relevant passages for a single query
qrels_unique_qids = qrels_df["qid"].nunique()
qrels_total_qids = len(qrels_df)
print(f"Number of unique qids in qrels: {qrels_unique_qids}")
print(f"Total number of qids in qrels: {qrels_total_qids}")

Number of unique qids in qrels: 3452
Total number of qids in qrels: 4201


In [32]:
# Check for queries that are not in the qrels set
queries_not_in_qrels = set(queries_df["qid"]) - set(qrels_df["qid"])
print(f"Number of queries not in qrels: {len(queries_not_in_qrels)}")

Number of queries not in qrels: 0


In [33]:
# Check for passagees that are in the qrels set but not in the passages set
passages_not_in_passages = set(qrels_df["pid"]) - set(passages_df["pid"])
print(f"Number of passages in qrels but not in passages: {len(passages_not_in_passages)}")

Number of passages in qrels but not in passages: 0


In [34]:
# Create a mapping from qid to relevant pids
qrels_mapping = defaultdict(list)

# Populate the mapping
for _, row in qrels_df.iterrows():
    qid = row["qid"]
    pid = row["pid"]
    qrels_mapping[qid].append(pid)

In [35]:
# For each query in queries_df, we want it to format as:
# {
#   "qid": "test0",
#   "query": "what is non controlling interest on balance sheet",
#   "qrels": ["doc1", "doc9"]
# }

queries_with_qrels = []

for _, row in queries_df.iterrows():
    qid = row["qid"]
    query_text = row["query"]
    relevant_pids = qrels_mapping[qid]
    
    queries_with_qrels.append({
        "qid": qid,
        "query": query_text,
        "qrels": relevant_pids
    })

In [36]:
# Check how a line looks like
print(queries_with_qrels[0])

{'qid': 'test0', 'query': 'what is non controlling interest on balance sheet', 'qrels': ['doc0', 'doc1']}


In [37]:
# Find the longest list of relevant pids
max_rels = max(len(qrels) for qrels in qrels_mapping.values())
print(f"Maximum number of relevant passages for any query: {max_rels}")

Maximum number of relevant passages for any query: 4


In [38]:
QUERIES_WITH_QRELS_OUTPUT_PATH = "../data/nq/processed/queries_with_qrels.jsonl"
# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(QUERIES_WITH_QRELS_OUTPUT_PATH), exist_ok=True)

# Save formatted data to be JSONL
with open(QUERIES_WITH_QRELS_OUTPUT_PATH, "w") as f:
    for item in queries_with_qrels:
        f.write(json.dumps(item) + "\n")

print(f"Saved formatted queries with qrels to {QUERIES_WITH_QRELS_OUTPUT_PATH}")

Saved formatted queries with qrels to ../data/nq/processed/queries_with_qrels.jsonl


In [39]:
# Get the set of `pids` that are not present in the `qrels` set
all_pids = set(passages_df["pid"])
relevant_pids = set(qrels_df["pid"])
non_relevant_pids = all_pids - relevant_pids
print(f"Total unique relevant pids: {len(relevant_pids)}")
print(f"Total unique non-relevant pids: {len(non_relevant_pids)}")

Total unique relevant pids: 4201
Total unique non-relevant pids: 2677267


In [40]:
# Randomly sample 156,000 non-relevant `pids`
# Set random seed for reproducibility
np.random.seed(42)
sampled_non_relevant_pids = set(
    np.random.choice(list(non_relevant_pids), size=196000, replace=False)
)

# Create a union of relevant pids and sampled non-relevant pids
final_pids = relevant_pids.union(sampled_non_relevant_pids)
print(f"Total pids after combining relevant and sampled non-relevant: {len(final_pids)}")

Total pids after combining relevant and sampled non-relevant: 200201


In [41]:
# Create a filtered passage collection dataframe with only the `final_pids`
filtered_passages_df = passages_df[passages_df["pid"].isin(final_pids)]

In [44]:
filtered_passages_df.head()

,pid,passage
0,doc0,"In accounting, minority interest (or non-contr..."
1,doc1,"It is, however, possible (such as through spec..."
6,doc6,"The fourth season of Chicago Fire, an American..."
10,doc10,"""Love Will Keep Us Alive"" is a song written by..."
11,doc11,Although the song was never formally released ...


In [42]:
# Load pre-trained embedding model
model = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
)

In [45]:
BATCH_SIZE = 64
PROCESSING_CHUNK_SIZE = 100000 # Process in batches to prevent GPU overloading
PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH = "../data/nq/processed/passages_with_embeddings.jsonl"

# Save the passages with embeddings to a JSONL file
# Process in chunks to avoid memory issues
with open(PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH, "w") as f:
    total_rows = len(filtered_passages_df)
    num_chunks = math.ceil(total_rows / PROCESSING_CHUNK_SIZE)

    print(f"Starting encoding for {total_rows} passages in {num_chunks} chunks...")

    for i in range(num_chunks):
        start_idx = i * PROCESSING_CHUNK_SIZE
        end_idx = min((i + 1) * PROCESSING_CHUNK_SIZE, total_rows) # ensure we don't go past end of DF

        print(f"Processing chunk {i + 1}/{num_chunks}, rows {start_idx} to {end_idx - 1}...")

        # Get the chunk from the DF
        chunk_df = filtered_passages_df.iloc[start_idx:end_idx]
        # Get data for this chunk
        passages = chunk_df["passage"].tolist()
        passage_ids = chunk_df["pid"].tolist()
        # Generate passage embeddings for this chunk
        embeddings = model.encode(
            passages, 
            batch_size=BATCH_SIZE, 
            show_progress_bar=False, 
            convert_to_numpy=False
        )

        # Write this chunk's records to the file immediately
        for pid, passage, embedding in zip(passage_ids, passages, embeddings):
            # Build a list of dictionaries where each row is:
            # {"pid": "0", "passage": "some text", "embedding": [0.0312, -0.0249, ...]}
            record = {
                "pid": str(pid),
                "passage": passage,
                "embedding": embedding.tolist()  # Convert numpy array to list for JSON serialization
            }
            f.write(json.dumps(record) + "\n")
        

print(f"Passages with embeddings saved to {PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH}")

Starting encoding for 200201 passages in 3 chunks...
Processing chunk 1/3, rows 0 to 99999...
Processing chunk 2/3, rows 100000 to 199999...
Processing chunk 3/3, rows 200000 to 200200...
Passages with embeddings saved to ../data/nq/processed/passages_with_embeddings.jsonl
